# Spatial Analysis: Natural Regrowth (NR) mitigation potential in Coca-Growing Areas

**Objective:**
This script assesses the Natural Regrowth mitigation potential of areas currently used for illegal coca cultivation in Colombia for 1km² grid cells where coca was detected in 2022.

**Workflow:**
1.  **Configuration:** Define spatial parameters (converting degrees to meters for the local latitude).
2.  **Preprocessing:** Reproject global yield rasters to the local UTM grid.
3.  **Extraction:** Clip data to the study area and extract yield values to the centroid of each grid cell.
4.  **Output:** Export a clean Excel table for statistical modeling.
   
   **Configuration:**
* **Coordinate System:** WGS 84 / UTM Zone 18N (EPSG: 32618), suitable for conformal projection analysis in Colombia.
* **Inputs:** * `sequestration_rate__mean__aboveground__full_extent__Mg_C_ha_yr.tif`  and `sequestration_error_ratio_layer_in_full_extent.tif` : Raw yield raster (Source: https://www.globalforestwatch.org/) Cook-Patton et al. 2020)

    * `Grillas1k_Coca_2022`: Vector grid of coca density. (Source: https://www.biesimci.org/)

In [1]:
import arcpy
from arcpy.sa import *
import math

# ===================================================================================
# 🟢 USER ACTION REQUIRED: DEFINE YOUR PATHS HERE
# ===================================================================================

# 1. Where is your Geodatabase? (This will be used for all intermediate processing)
# Example: r"C:\Users\Name\Project\MyProject.gdb"
workspace = r"M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb"

# 2. Where is your Input Raster? (The global yield TIF file)
# Example: r"C:\Users\Name\Downloads\prub200b_yld.tif"
input_raster = r"M:\Documents\Spatial_Analysis_Project\data\raw\sequestration_rate__mean__aboveground__full_extent__Mg_C_ha_yr.tif"
error_ratio = r"M:\Documents\Spatial_Analysis_Project\data\raw\sequestration_error_ratio_layer_in_full_extent.tif"

# 3. Where is your Shapefile? (The Coca_1km_2022 feature class)
# Example: r"C:\Users\Name\Project\MyProject.gdb\Coca_1km_2022"
shapefile = r"M:\Documents\Spatial_Analysis_Project\data\raw\Grillas1k_Coca_2022\Grillas1k.shp"

# 4. Where do you want the final Excel file?
output_excel = r"M:\Documents\Spatial_Analysis_Project\data\interim\Natural_Regrowth_Potential.xlsx"

# ===================================================================================
# END OF USER CONFIGURATION
# ===================================================================================

# Set Environment
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True
print(f"Workspace active: {workspace}")

Workspace active: M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb


## 1. Spatial Resolution Configuration

In this step, we define the spatial parameters for the analysis. The Area of Interest (AOI) is the density map of illegal coca cultivation for the year 2022 in Colombia represented in 1 km² and 5 km² grid cells. Colombia is located near the equator, requiring a conversion of the source raster resolution from degrees to meters to ensure accurate area calculations.

**Parameters:**
* **Latitude:** ~4.0° N (Centroid of Colombia).
* **Source Resolution:** 0.00833 degrees (Cell size X and Cell size Y).
* **Goal:** Calculate the equivalent cell size in meters to prepare for the projection.

In [2]:
# Constants for conversion
average_latitude = 4.0  # Centroid of Colombia
degree_to_meter_equator = 111000
original_cell_size_degrees = 0.00833

# Calculate conversion factor based on latitude
conversion_factor = degree_to_meter_equator * math.cos(math.radians(average_latitude))
cell_size_meters = original_cell_size_degrees * conversion_factor

print(f"Original Resolution: {original_cell_size_degrees} degrees")
print(f"Calculated Resolution: {cell_size_meters:.2f} meters")

# Define Output Coordinate System: WGS 84 / UTM zone 18N (EPSG: 32618)
out_coordinate_system = arcpy.SpatialReference(32618)

Original Resolution: 0.00833 degrees
Calculated Resolution: 922.38 meters


## 2. Reprojection and Clipping
We perform two critical spatial operations:
1.  **Project:** Transform the global raster to UTM Zone 18N using `NEAREST` neighbor resampling (to preserve original pixel values).
2.  **Clip:** Cut the raster to the exact bounding box of the Coca shapefile to reduce processing time.

In [3]:
# Define a dictionary to manage both input datasets
datasets = {
    "Mean_Potential": input_raster,
    "Error_Ratio": error_ratio
}

# Dictionary to store the paths of the processed rasters
processed_rasters = {}

# Check out Spatial Analyst Extension
arcpy.CheckOutExtension("Spatial")

try:
    # Get extent from the shapefile dynamically for clipping
    desc = arcpy.Describe(shapefile)
    extent = f"{desc.extent.XMin} {desc.extent.YMin} {desc.extent.XMax} {desc.extent.YMax}"
    
    for key, path in datasets.items():
        print(f"--- Processing {key} ---")
        
        # Define intermediate output names
        reproj_out = f"{workspace}\\{key}_reproj"
        clip_out = f"{workspace}\\{key}_clip"
        
        # Step 1: Projecting raster
        print(f"Step 1: Projecting {key}...")
        arcpy.management.ProjectRaster(
            path, 
            reproj_out, 
            out_coordinate_system, 
            resampling_type="NEAREST", 
            cell_size=cell_size_meters
        )

        # Step 2: Clipping raster
        print(f"Step 2: Clipping {key}...")
        arcpy.management.Clip(
            reproj_out, 
            extent, 
            clip_out, 
            "#", 
            -9999999, 
            "NONE", 
            "NO_MAINTAIN_EXTENT"
        )
        
        processed_rasters[key] = clip_out
        
    print("Preprocessing complete for both datasets.")

except Exception as e:
    print(f"Error during preprocessing: {e}")

--- Processing Mean_Potential ---
Step 1: Projecting Mean_Potential...
Step 2: Clipping Mean_Potential...
--- Processing Error_Ratio ---
Step 1: Projecting Error_Ratio...
Step 2: Clipping Error_Ratio...
Preprocessing complete for both datasets.


## 3. Extract Values to Points

To link the spatial data to the coca-growing sites, we sample the rasters at the geometric center of each cell. This avoids spatial bias and prepares the data for statistical modeling.

**Workflow:**
* **Centroids:** We convert the 1 km² polygon grids into centroid points.
* **Extraction:** We sample both the **Mean Sequestration** and the **Uncertainty Ratio** at each point location. This provides a specific yield potential and its associated error for every analyzed cell.

In [22]:
# Define output names
base_name = "NR_Coca_extension"
centroid_points = f"{workspace}\\{base_name}_centroids"

try:
    print("Step 1: Generating centroids from grid polygons...")
    arcpy.management.FeatureToPoint(shapefile, centroid_points, "CENTROID")

    print("Step 2: Extracting multiple raster values to points...")
    # Map the processed rasters to meaningful field names
    # processed_rasters["Mean_Potential"] and processed_rasters["Error_Ratio"] 
    # were created in the previous block
    extraction_list = [
        [processed_rasters["Mean_Potential"], "Mean_Seq_MgC"],
        [processed_rasters["Error_Ratio"], "Error_Ratio"]
    ]
    
    # Extract Multi Values to Points allows capturing both layers in one table
    arcpy.sa.ExtractMultiValuesToPoints(centroid_points, extraction_list, "NONE")
    
    print("Extraction complete. Fields 'Mean_Seq_MgC' and 'Error_Ratio' added to centroids.")

except Exception as e:
    print(f"Error during extraction: {e}")

Step 1: Generating centroids from grid polygons...
Step 2: Extracting multiple raster values to points...
Extraction complete. Fields 'Mean_Seq_MgC' and 'Error_Ratio' added to centroids.


# 4. Uncertainty Analysis: Intra-cell Variability

**Objective:**
To quantify sub-grid heterogeneity, we use a **cross-point sampling method**. This approach measures how much the raster-based data (e.g., yields or sequestration rates) varies within a single 1 km² grid cell.

**Methodology:**
1. **Geometric Offsets:** From the geometric centroid of each 1 km² cell, we generate four equidistant sampling points (North, South, East, West) at a distance of 0.5 km.
2. **Sampling:** Raster values are extracted at these four specific coordinates.
3. **Standard Deviation:** We calculate the standard deviation ($\sigma$) of these four values to provide a quantitative measure of spatial variability.

**Equation:**
$$\sigma = \sqrt{\frac{\sum_{i=1}^{4} (x_i - \bar{x})^2}{4}}$$

*Where:*
* $x_i$: Value at cross-point $i$.
* $\bar{x}$: Mean value of the four cross-points.

In [23]:
import arcpy
import os

# ===================================================================================
# 🟢 USER ACTION REQUIRED: DEFINE YOUR PATHS HERE
# ===================================================================================

# 1. Project Workspace
workspace = r"M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb"

# 2. Inputs
# Use the centroids generated in the previous analysis
centroids_input = f"{workspace}\\NR_1km_Coca_extension_centroids"
# The raster you want to test for variability (e.g., Carbon or Yield)
target_raster = f"{workspace}\\Mean_Potential_clip"

# 3. Output
output_excel_std = r"M:\Documents\Spatial_Analysis_Project\data\interim\NR_Spatial_Analysis.xlsx"

# 4. Grid ID Field
zone_id_field = "GRI_1K_F_1"

# ===================================================================================
# SETUP
# ===================================================================================
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True
spatial_ref = arcpy.Describe(centroids_input).spatialReference

## 5.1. Generating Spatial Sampling Points
We create four points per grid cell, offset by 500 meters from the center. This ensures the samples are representative of the cell's "corners" while remaining within the 1 km² boundary.

In [24]:
cross_points_output = f"{workspace}\\Coca_2022_Cross_Points"

# Create empty feature class
arcpy.management.CreateFeatureclass(workspace, "Coca_2022_Cross_Points", "POINT", spatial_reference=spatial_ref)
arcpy.management.AddField(cross_points_output, zone_id_field, "TEXT")
arcpy.management.AddField(cross_points_output, "Point_Type", "TEXT")

# 0.5 km distance in meters
distance = 500 

print("Calculating cross-points (N, S, E, W)...")
with arcpy.da.InsertCursor(cross_points_output, ["SHAPE@", zone_id_field, "Point_Type"]) as insert_cursor:
    with arcpy.da.SearchCursor(centroids_input, ["SHAPE@XY", zone_id_field]) as search_cursor:
        for row in search_cursor:
            centroid = row[0]
            grid_id = row[1]
            
            # Define offsets
            offsets = [
                (centroid[0], centroid[1] + distance, "N"),
                (centroid[0], centroid[1] - distance, "S"),
                (centroid[0] + distance, centroid[1], "E"),
                (centroid[0] - distance, centroid[1], "W")
            ]
            
            for x, y, p_type in offsets:
                new_point = arcpy.Point(x, y)
                insert_cursor.insertRow([arcpy.PointGeometry(new_point, spatial_ref), grid_id, p_type])

print(f"✅ Created 4 points per cell in: {cross_points_output}")

Calculating cross-points (N, S, E, W)...
✅ Created 4 points per cell in: M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb\Coca_2022_Cross_Points


## 5.2. Calculate Intra-cell Variability Statistics

In this step, we quantify the spatial heterogeneity within each grid cell. 

**Workflow:**
1.  **Extraction:** We sample the target raster (Natural Regrowth Potential) at the four cross-points (N, S, E, W) generated in the previous step.
2.  **Summary Statistics:** We use the `Statistics` tool to calculate the **Standard Deviation** of these samples for each unique Grid ID. This provides a single metric (`STD_Variability`) representing the uncertainty for that specific cell.

In [26]:
# Output names
points_with_values = f"{workspace}\\NR_Cross_Points_Extracted"
stats_table = f"{workspace}\\NR_Variability_Stats"

# Field used for grouping statistics (Unique Grid ID)
join_field_id = "GRI_1K_F_1"

try:
    arcpy.CheckOutExtension("Spatial")
    
    print("Step 1: Extracting raster values to cross-points...")
    # Extract raster values to the cross-points feature class
    arcpy.sa.ExtractValuesToPoints(cross_points_output, target_raster, points_with_values)
    
    print("Step 2: Calculating Standard Deviation per Grid ID...")
    # Calculate STD of the 'RASTERVALU' field, grouped by Grid ID
    # This creates a standalone table with [Grid_ID, STD_RASTERVALU]
    arcpy.analysis.Statistics(
        in_table=points_with_values, 
        out_table=stats_table, 
        statistics_fields=[["RASTERVALU", "STD"]], 
        case_field=join_field_id
    )
    
    print(f"✅ Statistics calculated. Table created at: {stats_table}")

except Exception as e:
    print(f"❌ Error during statistics calculation: {e}")

finally:
    arcpy.CheckInExtension("Spatial")

Step 1: Extracting raster values to cross-points...
Step 2: Calculating Standard Deviation per Grid ID...
✅ Statistics calculated. Table created at: M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb\NR_Variability_Stats


## 5.3. Data Integration and Final Export

Finally, we consolidate all the calculated parameters into a single dataset for Phase 2 analysis.

**Workflow:**
1.  **Join Statistics:** We join the calculated **Standard Deviation** (`STD_RASTERVALU`) back to the original Centroids feature class.
2.  **Verify Fields:** The centroids now contain:
    * **Mean Sequestration** (from Block 3)
    * **Error Ratio** (from Block 3)
    * **Coca Density** (Original Shapefile Attribute)
    * **Intra-cell Variability** (Newly joined STD)
3.  **Export:** We export this comprehensive attribute table to Excel.

In [27]:
# Reference to the centroids (which already hold Mean & Error from previous steps)
centroid_points = f"{workspace}\\NR_Coca_extension_centroids"

# The Excel file path defined in Block 1
# output_excel = ... (Already defined as 'Natural_Regrowth_Potential.xlsx')

try:
    print(f"Step 1: Joining Variability Statistics to Centroids...")
    
    # Join the stats table (Block 5.2) to the centroids (Block 3)
    arcpy.management.JoinField(
        in_data=centroid_points, 
        in_field=join_field_id, 
        join_table=stats_table, 
        join_field=join_field_id, 
        fields=["STD_RASTERVALU"] # Only join the STD column
    )
    
    # Optional: Rename the field alias for clarity in Excel
    # Note: The actual field name might remain 'STD_RASTERVALU' depending on GDB limits
    try:
        arcpy.management.AlterField(centroid_points, "STD_RASTERVALU", "STD_Variability", "Intra-cell Variability")
        print("Field renamed to 'STD_Variability'.")
    except:
        print("Field rename skipped (Field might be locked or unsupported). Keeping 'STD_RASTERVALU'.")

    # --- Final Export ---
    print(f"Step 2: Exporting final integrated table to: {output_excel}")
    arcpy.conversion.TableToExcel(centroid_points, output_excel)
    
    print("✅ SUCCESS: Final dataset exported. Ready for Phase 2 Python Analysis.")

except Exception as e:
    print(f"❌ Error during export: {e}")

Step 1: Joining Variability Statistics to Centroids...
Field renamed to 'STD_Variability'.
Step 2: Exporting final integrated table to: M:\Documents\Spatial_Analysis_Project\data\interim\Natural_Regrowth_Potential.xlsx
✅ SUCCESS: Final dataset exported. Ready for Phase 2 Python Analysis.
